# Verificación A5: código muerto
Ejecución de la lista de verificación manual del README sobre la interfaz de widgets, después de eliminar los métodos muertos. Cada paso usa los mismos botones de la interfaz.

In [1]:
import sys, json, re
sys.path.insert(0, "src")

from minichat.ui.chat_app import ChatApp
app = ChatApp("echo")
v = app.view
app.show()

In [2]:
def estado():
    return re.sub(r"<[^>]+>", "", v.status.value)

def enviar(rol, texto):
    v.role.value = rol
    v.text.value = texto
    v.btn_send.click()

def mostrar():
    print("Estado:", estado())
    for etiqueta, _ in v.sel_index.options:
        print("  ", etiqueta)

## 1. Enviar un mensaje con rol `user`

In [3]:
enviar("user", "hola, ¿cómo estás?")
mostrar()

Estado: Done.
   0: [02:08:20] user: hola, ¿cómo estás?
   1: [02:08:20] assistant: I see. "hola, ¿cómo estás?" is…


## 2. Agregar un mensaje `assistant` y otro `system`

In [4]:
enviar("assistant", "respuesta escrita a mano")
enviar("system", "eres un bot amable")
mostrar()

Estado: Messages updated.
   0: [02:08:20] user: hola, ¿cómo estás?
   1: [02:08:20] assistant: I see. "hola, ¿cómo estás?" is…
   2: [02:08:20] assistant: respuesta escrita a mano
   3: [02:08:20] system: eres un bot amable


## 3. Mover los sliders y enviar otro mensaje

In [5]:
v.temp.value, v.topp.value, v.topk.value, v.maxnt.value = 1.0, 0.5, 10, 64
enviar("user", "otro mensaje")
print(v._ctl.config)
mostrar()

GenerationConfig(temperature=1.0, top_p=0.5, top_k=10, max_new_tokens=64, max_ctx_tokens=900)
Estado: Done.
   0: [02:08:20] user: hola, ¿cómo estás?
   1: [02:08:20] assistant: I see. "hola, ¿cómo estás?" is…
   2: [02:08:20] assistant: respuesta escrita a mano
   3: [02:08:20] system: eres un bot amable
   4: [02:08:20] user: otro mensaje
   5: [02:08:20] assistant: Interesting. Why do you think …


## 4. Seleccionar un mensaje y borrarlo con **Delete selected**

In [6]:
v.sel_index.value = 0
v.btn_delete.click()
mostrar()

Estado: Messages updated.
   0: [02:08:20] assistant: I see. "hola, ¿cómo estás?" is…
   1: [02:08:20] assistant: respuesta escrita a mano
   2: [02:08:20] system: eres un bot amable
   3: [02:08:20] user: otro mensaje
   4: [02:08:20] assistant: Interesting. Why do you think …


## 5. **Export JSON** (usa `as_public()` en lugar de `get_summary()`)

In [7]:
v.btn_export.click()
print("Estado:", estado())
with open("chat_transcript.json", encoding="utf-8") as fh:
    for m in json.load(fh):
        print(f"{m['role']:>9}: {m['content']}  meta={sorted(m['meta'])}")

Estado: Exported: chat_transcript.json
assistant: I see. "hola, ¿cómo estás?" is a good point.  meta=['backend', 'gen_cfg', 'latency_ms', 'prompt_chars', 'reply_chars']
assistant: respuesta escrita a mano  meta=[]
   system: eres un bot amable  meta=[]
     user: otro mensaje  meta=[]
assistant: Interesting. Why do you think "otro mensaje"?  meta=['backend', 'gen_cfg', 'latency_ms', 'prompt_chars', 'reply_chars']


## 6. **Delete ALL**

In [8]:
v.btn_clear_all.click()
mostrar()
print("Mensajes en la sesión:", len(v._session.all()))

Estado: Messages updated.
Mensajes en la sesión: 0


## Los métodos eliminados ya no existen

In [9]:
from minichat.model.chat_session import ChatSession
from minichat.controller.chat_controller import ChatController
from minichat.model.role import Role

for clase, metodo in [(ChatSession, "get_summary"), (ChatSession, "get_observer_count"),
                      (ChatSession, "has_observers"), (ChatController, "get_observer_count"),
                      (ChatController, "has_observers"), (Role, "list")]:
    print(f"{clase.__name__}.{metodo}:", "existe" if hasattr(clase, metodo) else "eliminado")

ChatSession.get_summary: eliminado
ChatSession.get_observer_count: eliminado
ChatSession.has_observers: eliminado
ChatController.get_observer_count: eliminado
ChatController.has_observers: eliminado
Role.list: eliminado
